In [1]:
from pathlib import Path

import numpy as np
import time
import pickle
import sys
from tqdm.notebook import tqdm
import itertools

from SolverV9 import (
    LiouvilleSpectroscopySolver,
    SpectroscopyPlotter,
    standard_nq_protocol,
)

from IPython.display import display, HTML
display(HTML("<style>div.output_area pre, div.jp-OutputArea-output pre {white-space: pre;}</style>"))

np.set_printoptions(threshold=sys.maxsize, linewidth=np.inf, precision=5, suppress=True)

### Coupled Model

### Solver Parameters

- `T`: System temperature. `T = 0` assumes that the system initially occupies its ground state.
- `Eta`: Homogeneous broadening used in the resolvents, in eV. It primarily controls the spectral linewidth.
- `backend`: Numerical implementation used by the solver. `"dense"` uses dense matrices and NumPy linear-algebra routines.

In [2]:
solver_params = {
    "T": 0.0,
    "Eta": 0.000001,
    "backend": "dense",
    "parallel_backend": "threading",
    "n_jobs": -1,
}

In [3]:
hbar = 0.658211951  # in eV fs

# parameters
en_x=1.4907 # dark state excitonic resonant energy in eV
detuning = -0.002 # detuning between exciton and cavity in eV
en_cav=en_x + detuning # cavity resonant energy in eV

order = 3

rabi = 0.003  # vaccum rabi splitting
g = rabi / 2  # coupling strength for light-matter interaction in eV
J = 0.000 # coupling strength between XX and XXb
n_th = 0. # average number of cavity photons

mu_qm1 = 0.3 # dipole strength for C
mu_qm2 = 0.29  # dipole strength for CC
mu_x1 = 0.04  # dipole strength for X
mu_x2 = 0.25  # dipole strength for 2Q X

kappa = 0.0011 # cavity decay strength
gamma_x_decay = 0.0001  # exciton decay strength
gamma_cav_phase = 0.0006 / 10 # cavity dephasing strength
gamma_exc_phase = 0.0001 / 10 # exciton dephasing strength
gamma_xc = 0.001 # x-c swapping strength
gamma_cx = 0.001 # c-x swapping strength

xxb_bind=0.00086 # strength of DD biexciton binding

grid = 100
t2 = 50
model = "rw"  # "no_rw" for no rotating wave approximation, "rw" for rotating wave approximation
# model="no-rw"

date = time.strftime('%Y-%m-%d')
hour = time.strftime("%Hh%Mm%S")

### 0-quantum + 1-quantum + 2-quantum
loop over i for any parameter

In [ ]:
param = "test g"
rang = ""
rang = [0.0002,0.0008,10]
# kappa_list = [0.0011, 0.0015, 0.0019]
for i in tqdm(np.linspace(*rang)):
# for i in tqdm(np.geomspace(*rang)):
# for i in tqdm(list(itertools.product(np.linspace(*rang), np.linspace(*rang)))):
# for i in tqdm(list(itertools.product(np.linspace(*rang), np.linspace(*rang), np.linspace(*rang)))):
# for i in tqdm(list(itertools.product(np.linspace(*rang), np.linspace(*rang), np.linspace(rang[0], rang[1], 6), np.linspace(*rang)))):
# for i in [1.]:
#     mu_qm1 = i[0]
#     mu_qm2 = i[1]
#     mu_d1 = i[2]
#     mu_d2 = i[3]
    g = i
#     gamma_cav_phase = i[2]
#     en_cav=en_b + detuning
#     print(param_name)


    # setting up dm
    exc_basis = np.zeros((7, 7))
    exc_basis[0, 0] = 1 # ground,X,C,XX,XC,CC,XXb
    rho = exc_basis @ exc_basis.T  # ground state of Hamiltonian

    wc = en_cav / hbar
    wx = en_x / hbar  # exciton resonant frequencies

    # each individual lowering operators
    x_low = np.zeros_like(exc_basis)
    x_low[0, 1] = 1

    c_low = np.zeros_like(exc_basis)
    c_low[0, 2] = 1

    xx_low = np.zeros_like(exc_basis)
    xx_low[1, 3] = 1

    xc_low = np.zeros_like(exc_basis)
    xc_low[1, 4] = 1 / np.sqrt(2)

    cx_low = np.zeros_like(exc_basis)
    cx_low[2, 4] = 1 / np.sqrt(2)

    cc_low = np.zeros_like(exc_basis)
    cc_low[2, 5] = 1

    xxb_low = np.zeros_like(exc_basis)
    xxb_low[1, 6] = 1

    # global lowering operators
    cav_low = c_low + xc_low + cc_low
    exc_low = x_low + xx_low + cx_low + xxb_low

    H0_x = x_low.T @ x_low  # x-pop
    H0_c = c_low.T @ c_low  # c-pop
    H0_xx = xx_low.T @ xx_low  # xx-pop
    H0_xc = xc_low.T @ xc_low + cx_low.T @ cx_low  # xc-pop
    H0_cc = cc_low.T @ cc_low  # cc-pop
    H0_xxb = xxb_low.T @ xxb_low  # xxb-pop

    H0 = (en_x * H0_x + en_cav * H0_c + 2 * en_x * H0_xx +
          (en_x + en_cav) * H0_xc + 2 * en_cav * H0_cc + (2 * en_x - xxb_bind) * H0_xxb)

    H_int = x_low.T @ c_low  # 1Q light-matter coupling
    H_int += 2 * (xx_low.T @ xc_low + xxb_low.T @ xc_low + cx_low.T @ cc_low) # 2Q bright light-matter coupling
    if model == "no-rw":
        H_int += 2 * cav_low @ exc_low  # 2Q non-RWA terms (don't conserve energy)
    elif model != "rw":
        print('Wrong model name')
    H_int += H_int.T

    H_coupling = xx_low.T @ xxb_low
    H_coupling += H_coupling.T

    H = H0 + g * H_int + J * H_coupling  # total hamiltonian
    # print("H0", H0)
    # print("H_int\n", H_int)
    # print("H_coupling\n", H_coupling)
    # print("H", H)

    en, T = np.linalg.eig(H)
    D = T[:, ::-1]
    # print(en)
    # print("D\n", D)

    ad_temp = np.zeros((7, 7))
    for l in range(1, 3):
        ad_temp[0][l] = np.real(mu_qm1 * D[2][l] + mu_x1 * D[1][l])
    for k in range(3, 7):
        ad_temp[1][k] = np.real(np.sqrt(2) * D[2][1] * mu_qm2 * D[5][k] + D[1][1] * mu_qm2 * D[4][k])
        ad_temp[2][k] = np.real(np.sqrt(2) * D[1][2] * mu_x2 * D[6][k] + D[2][2] * mu_qm1 * D[4][k])
        ad_temp[3][k] = np.real(np.sqrt(2) * D[2][3] * mu_qm2 * D[5][k] + D[1][3] * mu_qm2 * D[4][k])

    ad_temp2 = np.zeros((7, 7))
    for l in range(1, 3):
        ad_temp2[0][l] = np.real(mu_qm1 * D[2][l] + mu_x1 * D[1][l])
    for k in range(3, 7):
        ad_temp2[1][k] = np.real(D[1][1] * (mu_qm2 * D[4][k] + np.sqrt(2) * mu_x2 * (D[3][k] + D[6][k])) + D[2][1] * (np.sqrt(2) * mu_qm2 * D[5][k] + mu_x2 * D[4][k]))
        ad_temp2[2][k] = np.real(D[1][2] * (mu_qm2 * D[4][k] + np.sqrt(2) * mu_x2 * (D[3][k] + D[6][k])) + D[2][2] * (np.sqrt(2) * mu_qm2 * D[5][k] + mu_x2 * D[4][k]))


    ad = D @ ad_temp2 @ D.T
    mu = ad.T + ad

    # ad = mu_qm2 * cav_low + (mu_qm1 - mu_qm2) * c_low + mu_d2 * exc_d_low + (mu_d1 - mu_d2) * d_low  # lowering operator
    # ad_paper = mu_qm2 * (cc_low + bc_low) + mu_d2 * xxb_low + mu_d1 * d_low + mu_qm1 * c_low
    # obs = ad_paper + ad_paper.dag()
    print("mud\n", mu)

    # collapse operators
    c_cav_rel = np.sqrt(kappa * (n_th + 1)) * cav_low # cavity relaxation
    c_cav_exc = np.sqrt(kappa * n_th) * cav_low.T # cavity excitation
    c_x_rel = np.sqrt(gamma_x_decay * (n_th + 1)) * exc_low # exciton relaxation
    c_x_exc = np.sqrt(gamma_x_decay * n_th) * exc_low.T # exciton excitation

    c_c_rel = np.sqrt(kappa * (n_th + 1)) * c_low # cavity relaxation
    c_c_exc = np.sqrt(kappa * n_th) * c_low.T # cavity excitation
    c_x__rel = np.sqrt(gamma_x_decay * (n_th + 1)) * x_low # exciton relaxation
    c_x__exc = np.sqrt(gamma_x_decay * n_th) * x_low.T # exciton excitation
    c_cc_rel = np.sqrt(kappa * (n_th + 1)) * cc_low # cavity relaxation
    c_cc_exc = np.sqrt(kappa * n_th) * cc_low.T # cavity excitation
    c_xx_rel = np.sqrt(gamma_x_decay * (n_th + 1)) * xx_low # exciton relaxation
    c_xx_exc = np.sqrt(gamma_x_decay * n_th) * xx_low.T # exciton excitation
    c_cx_rel = np.sqrt(kappa * (n_th + 1)) * cx_low # cavity relaxation
    c_cx_exc = np.sqrt(kappa * n_th) * cx_low.T # cavity excitation
    c_xc_rel = np.sqrt(gamma_x_decay * (n_th + 1)) * xc_low # exciton relaxation
    c_xc_exc = np.sqrt(gamma_x_decay * n_th) * xc_low.T # exciton excitation
    c_xxb_rel = np.sqrt(gamma_x_decay * (n_th + 1)) * xxb_low # exciton relaxation
    c_xxb_exc = np.sqrt(gamma_x_decay * n_th) * xxb_low.T # exciton excitation

    c_x_dep = np.sqrt(gamma_exc_phase) * en_x * H0_x # X pop
    c_c_dep = np.sqrt(gamma_cav_phase) * en_cav * H0_c # C pop
    c_xx_dep = np.sqrt(gamma_exc_phase) * 2 * en_x * H0_xx # XX pop
    c_xc_dep = (np.sqrt(gamma_cav_phase) + np.sqrt(gamma_exc_phase)) * (en_x + en_cav) / 2 * H0_xc # XC pop
    c_cc_dep = np.sqrt(gamma_cav_phase) * 2 * en_cav * H0_cc # CC pop
    c_xxb_dep = np.sqrt(gamma_exc_phase) * (2 * en_x - xxb_bind ) * H0_xxb # XXb pop

    c_x_cav = np.sqrt(gamma_xc) * cav_low.T @ exc_low # bright exciton to cavity photon transition
    c_cav_x = np.sqrt(gamma_xc) * exc_low.T @ cav_low # cavity photon to bright exciton transition

    c_ops = [(c_cav_rel,1), (c_cav_exc,1), (c_x_rel,1), (c_x_exc,1), (c_x_dep,1), (c_c_dep,1), (c_xx_dep,1), (c_xc_dep,1), (c_cc_dep,1), (c_xxb_dep,1), (c_cav_x,1), (c_x_cav,1)]
    c_ops = [(c_c_rel,1), (c_c_exc,1), (c_x_rel,1), (c_x_exc,1), (c_cc_rel,1), (c_cc_exc,1), (c_xxb_rel,1), (c_xxb_exc,1), (c_xx_rel,1), (c_xx_exc,1), (c_cx_rel,1), (c_cx_exc,1), (c_xc_rel,1), (c_xc_exc,1), (c_x_dep,1), (c_c_dep,1), (c_xc_dep,1), (c_cc_dep,1), (c_xxb_dep,1)]
    # for j in c_ops:
    #     print()
    #     print(j)

    # solver
    solver = LiouvilleSpectroscopySolver(solver_params)
    solver.feed_model(
        H,
        mu,
        c_ops_raw=c_ops,
        initial_density_matrix=rho,
        density_matrix_basis="site",
    )

    # pathways
    pathways = solver.generate_pathways_with_ufss(
        "-++",
        maximum_manifold=2,
        component="chi3_1q",
    )

    protocol = standard_nq_protocol(
        order=1,
        nq_interval=1,
        detection_interval=3,
        n_interactions=3,
        nq_axis="omega_1q",
        detection_axis="omega_emit",
    )

    [(pathway.name, pathway.interactions, pathway.coherence_orders) for pathway in pathways]

    # generate the spectrum
    omega_1q = np.linspace(-1.495, -1.482, grid)
    omega_emit = np.linspace(1.482, 1.495, grid)

    delays = {
        "t2": t2 / hbar
    }

    result = solver.generate_NQ_spectrum(
        1,
        protocol,
        axes={"omega_1q": omega_1q, "omega_emit": omega_emit},
        delays=delays,
        pathways=pathways
        )

    save_pdf = True
    output_directory = Path.cwd() / "Result_Test" / "Multiorder_pathway_plot" / date / f"{hour}_{param}{rang}"
    output_directory.mkdir(parents=True, exist_ok=True)

    plotter = SpectroscopyPlotter(detection_phase=np.pi/2)
    plot_result = plotter.plot_pathways_multiorder(
        result,
        pathways="all",
        totals=["1Q"],
        view="real",
        normalization="none",
        axis_labels={
            "omega_1q": "1Q energy (eV)",
            "omega_emit": "Emission energy (eV)",
        },
        include_diagrams=False,
        display_diagrams=False,
        save_pdf=save_pdf,
        output_directory=output_directory if save_pdf else None,
        spectrum_pdf_name=f"chi3_Q1_real_{param}{i:.6f}.png",
        # spectrum_pdf_name=f"chi3_Q1_real_{param}{param_name:.4f}_{param_name2:.4f}.png",
        # spectrum_pdf_name=f"chi3_Q1_real_{param}{i[0]:.3f}_{i[1]:.3f}_{i[2]:.3f}.png",
        # spectrum_pdf_name=f"chi3_Q1_real_{param}{i[0]:.3f}_{i[1]:.3f}_{i[2]:.3f}_{i[3]:.3f}.png",
        show=False,
    )

    plot_result = plotter.plot_pathways_multiorder(
        result,
        pathways="all",
        totals=["1Q"],
        view="imag",
        normalization="none",
        axis_labels={
            "omega_1q": "1Q energy (eV)",
            "omega_emit": "Emission energy (eV)",
        },
        include_diagrams=False,
        display_diagrams=False,
        save_pdf=save_pdf,
        output_directory=output_directory if save_pdf else None,
        spectrum_pdf_name=f"chi3_Q1_imag_{param}{i:.6f}.png",
        # spectrum_pdf_name=f"chi3_Q1_imag_{param}{param_name:.4f}_{param_name2:.4f}.png",
        # spectrum_pdf_name=f"chi3_Q1_imag_{param}{i[0]:.3f}_{i[1]:.3f}_{i[2]:.3f}.png",
        # spectrum_pdf_name=f"chi3_Q1_imag_{param}{i[0]:.3f}_{i[1]:.3f}_{i[2]:.3f}_{i[3]:.3f}.png",
        show=False,
    )

    plot_result = plotter.plot_pathways_multiorder(
        result,
        pathways="all",
        totals=["1Q"],
        view="abs",
        normalization="none",
        axis_labels={
            "omega_1q": "1Q energy (eV)",
            "omega_emit": "Emission energy (eV)",
        },
        include_diagrams=False,
        display_diagrams=False,
        save_pdf=save_pdf,
        output_directory=output_directory if save_pdf else None,
        spectrum_pdf_name=f"chi3_Q1_abs_{param}{i:.6f}.png",
        # spectrum_pdf_name=f"chi3_Q1_abs_{param}{param_name:.4f}_{param_name2:.4f}.png",
        # spectrum_pdf_name=f"chi3_Q1_abs_{param}{i[0]:.3f}_{i[1]:.3f}_{i[2]:.3f}.png",
        # spectrum_pdf_name=f"chi3_Q1_abs_{param}{i[0]:.3f}_{i[1]:.3f}_{i[2]:.3f}_{i[3]:.3f}.png",
        show=False,
    )

    # plotter.plot_spectrum_result_contours(
    #     result,
    #     spectra="pathways",
    #     names=["R1", "R2", "R3"],
    #     totals="selected",
    #     normalization="panel",
    #     color_map="PuOr",
    #     levels=25,
    #     show=False,
    # )
    print(f"{i} diagram done\n")

  0%|          | 0/10 [00:00<?, ?it/s]

mud
 [[ 0.     +0.j  0.04   +0.j  0.3    +0.j  0.     +0.j -0.     +0.j  0.     +0.j  0.     +0.j]
 [ 0.04   +0.j -0.     +0.j -0.     +0.j  0.35355+0.j  0.29   +0.j  0.     +0.j  0.35355+0.j]
 [ 0.3    +0.j -0.     +0.j  0.     +0.j -0.     +0.j  0.25   +0.j  0.41012+0.j  0.     +0.j]
 [ 0.     +0.j  0.35355+0.j -0.     +0.j  0.     +0.j  0.     +0.j  0.     +0.j  0.     +0.j]
 [-0.     +0.j  0.29   +0.j  0.25   +0.j  0.     +0.j -0.     +0.j  0.     +0.j -0.     +0.j]
 [ 0.     +0.j  0.     +0.j  0.41012+0.j  0.     +0.j  0.     +0.j  0.     +0.j  0.     +0.j]
 [ 0.     +0.j  0.35355+0.j  0.     +0.j  0.     +0.j -0.     +0.j  0.     +0.j  0.     +0.j]]
--- Model loading ---
Model transformed to the eigenbasis.
Liouville backend ready: dense.
Calculating 3 pathway spectrum/s on a 100x100 grid with protocol 'standard_1q' using parallel=threading, n_jobs=-1.
Using dense prefix-tree pathway reuse.
0.0002 diagram done

mud
 [[ 0.     +0.j  0.04   +0.j  0.3    +0.j  0.     +0.j -0.     +0

In [ ]:
with open(f"{output_directory}/parameters.txt", "w", encoding="utf-8") as file:
    file.write(hour+"\n")
    file.write(f"Energies  exciton={en_x:.5f}, cavity={en_cav:.5f}\n")
    file.write(f"detuning delta={detuning:.4}\n")
    file.write(f"light-matter coupling g={g:.4f}\n")
    file.write(f"dephasing  excitons={gamma_exc_phase:.4f}, cavity={gamma_cav_phase:.4f}\n")
    file.write(f"decay  exciton={gamma_x_decay:.4f}, cavity={kappa:.4f}\n")
    file.write(f"binding energies ={xxb_bind:.4f}\n")
    file.write(f"swapping strength  exciton-cavity={gamma_xc:.4f}, cavity-exciton={gamma_cx:.4f}\n")
    file.write(f"broadening eta={solver_params["Eta"]}\n")
    file.write(f"grid size(f1,f3)={grid}\n")
    file.write(f"population time={t2}\n")
    file.write(f"average photons={n_th}\n")
    file.write(f"model={model}\n")

In [ ]:
# print("Plotted panels:", plot_result.panel_names)
# print("Matching UFSS diagrams:", tuple(plot_result.diagrams))
# if save_pdf:
#     print("Spectrum PDF:", plot_result.spectrum_pdf)
#     print("Diagram PDFs:", plot_result.diagram_paths)
